# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/Vaanya0730/flyrank-ml-internship-starter.git /content/flyrank-ml-internship-starter
import os
import sys
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

# Locate the repository when running in Colab.
REPO = Path("/content/flyrank-ml-internship-starter")
if not REPO.exists():
    REPO = Path.cwd()

sys.path.insert(0, str(REPO / "scripts"))

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

DATA_PATH = REPO / "data" / "raw" / "content_refresh_anonymized.csv"
FEATURE_PATH = REPO / "data" / "processed" / "refresh_feature_vector.csv"
BASELINE_PATH = REPO / "data" / "processed" / "baseline_refresh_queue.csv"

# Create the prepared feature vector and baseline if they do not already exist.
if not FEATURE_PATH.exists():
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "01_prepare_features.py")],
        check=True,
    )

if not BASELINE_PATH.exists():
    subprocess.run(
        [sys.executable, str(REPO / "scripts" / "02_baseline_score.py")],
        check=True,
    )

frame = pd.read_csv(FEATURE_PATH)
baseline = pd.read_csv(BASELINE_PATH)

print("Prepared rows:", len(frame))
print("Baseline rows:", len(baseline))
print("Declining rate:", round(frame["is_declining_label"].mean(), 4))


fatal: destination path '/content/flyrank-ml-internship-starter' already exists and is not an empty directory.
Prepared rows: 30000
Baseline rows: 30000
Declining rate: 0.5421


## Method choice

I use Logistic Regression as the primary model because the task is a binary yes/no prediction: whether a content item is declining.

Logistic Regression is a good first model because it is simple and interpretable. I also compare it with a Random Forest to check whether a more flexible model improves the ranking of declining pages.

The target is `is_declining_label`. I do not use `trend_direction` or `trend_pct` as features because they are used to construct the target and would leak the answer.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the model feature matrix.
numeric_features = [
    col for col in MODEL_NUMERIC_FEATURES
    if col in frame.columns
]

categorical_features = [
    col for col in MODEL_CATEGORICAL_FEATURES
    if col in frame.columns
]

numeric_frame = (
    frame[numeric_features]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

categorical_frame = (
    frame[categorical_features]
    .fillna("unknown")
    .astype(str)
)

encoded_frame = pd.get_dummies(
    categorical_frame,
    prefix=categorical_features,
    dtype=float
)

X = pd.concat(
    [
        numeric_frame.reset_index(drop=True),
        encoded_frame.reset_index(drop=True),
    ],
    axis=1
)

y = frame["is_declining_label"].astype(int)

# Client-aware 80/20 split.
all_indices = np.arange(len(frame))
client_series = frame["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

split_strategy = "stratified_row_holdout"

if len(unique_clients) >= 5:
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled_clients = rng.permutation(unique_clients)

    test_client_count = max(1, int(round(len(shuffled_clients) * 0.20)))
    test_clients = set(shuffled_clients[:test_client_count])

    test_mask = client_series.isin(test_clients).to_numpy()

    train_idx = all_indices[~test_mask]
    test_idx = all_indices[test_mask]

    if (
        len(train_idx) > 0
        and len(test_idx) > 0
        and y.iloc[train_idx].nunique() == 2
        and y.iloc[test_idx].nunique() == 2
    ):
        split_strategy = "client_holdout"

if split_strategy != "client_holdout":
    train_idx, test_idx = train_test_split(
        all_indices,
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=y
    )

train_X = X.iloc[train_idx]
test_X = X.iloc[test_idx]
train_y = y.iloc[train_idx]
test_y = y.iloc[test_idx]

print("Split strategy:", split_strategy)
print("Train rows:", len(train_idx))
print("Test rows:", len(test_idx))
print("Train positive rate:", round(train_y.mean(), 4))
print("Test positive rate:", round(test_y.mean(), 4))

Split strategy: client_holdout
Train rows: 27675
Test rows: 2325
Train positive rate: 0.5548
Test positive rate: 0.391


## Split design

I use a client-aware holdout rather than randomly splitting rows.

Approximately 20% of clients are held out for testing, so pages from the same client do not appear in both train and test. This is a more honest estimate of performance on unseen clients and matches the split design used by the repository's reference training pipeline.

If the client holdout cannot produce both target classes in train and test, I fall back to a stratified row split.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,      # <-- add this
    roc_auc_score,
)

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        max_depth=5,
        min_samples_leaf=50,
        random_state=RANDOM_STATE,
    ),

    "Random Forest": RandomForestClassifier(
        class_weight="balanced_subsample",
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
}

def get_metrics(y_true, scores):
    predictions = (scores >= 0.5).astype(int)

    return {
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1": f1_score(y_true, predictions, zero_division=0),
        "Precision@20": precision_at_k(y_true, scores, 20),
        "Precision@50": precision_at_k(y_true, scores, 50),
        "Precision@100": precision_at_k(y_true, scores, 100),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "Average Precision": average_precision_score(y_true, scores),
    }

# Baseline scores for exactly the same test rows.
baseline_lookup = baseline.set_index("content_id")["baseline_refresh_score"]

baseline_scores = (
    frame.iloc[test_idx]["content_id"]
    .map(baseline_lookup)
    .fillna(0)
    .to_numpy()
)

results = []

baseline_metrics = get_metrics(test_y, baseline_scores)
results.append({
    "Method": "Week-4 Baseline",
    **baseline_metrics
})

fitted_models = {}
model_scores = {}

for name, model in models.items():
    model.fit(train_X, train_y)

    scores = model.predict_proba(test_X)[:, 1]

    fitted_models[name] = model
    model_scores[name] = scores

    metrics = get_metrics(test_y, scores)

    results.append({
        "Method": name,
        **metrics
    })

comparison = pd.DataFrame(results)

comparison = comparison.sort_values(
    ["Precision@50", "Average Precision", "ROC-AUC"],
    ascending=False
).reset_index(drop=True)

display(comparison.round(4))

best_model_name = comparison.loc[
    comparison["Method"] != "Week-4 Baseline",
    "Method"
].iloc[0]

print("Best learned model by Precision@50:", best_model_name)

,Method,Accuracy,Precision,Recall,F1,Precision@20,Precision@50,Precision@100,ROC-AUC,Average Precision
0,Random Forest,0.6723,0.5610,0.7437,0.6395,0.65,0.74,0.72,0.7500,0.6182
1,Decision Tree,0.6766,0.5686,0.7162,0.6339,0.45,0.58,0.62,0.7415,0.5753
2,Logistic Regression,0.6606,0.5659,0.5666,0.5662,0.35,0.40,0.44,0.7003,0.5215
3,Week-4 Baseline,0.6086,0.4986,0.1892,0.2743,0.15,0.24,0.36,0.6269,0.4676


Best learned model by Precision@50: Random Forest


## Train and compare against the baseline

I train Logistic Regression, a shallow Decision Tree, and Random Forest.

The models are evaluated on the same test set used for the baseline. Because this is a ranking-oriented question, Precision@50 is the main comparison metric, with Precision@20 and Precision@100 also reported.

The baseline uses the repository's deterministic `baseline_refresh_score`, while the learned models use predicted probability of decline.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
best_scores = model_scores[best_model_name]

error_df = frame.iloc[test_idx][
    [
        "content_id",
        "client_id",
        "is_declining_label",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        "ctr",
        "avg_position",
        "days_since_last_update",
        "content_age_days",
        "word_count",
    ]
].copy()

error_df["predicted_probability"] = best_scores
error_df["predicted_label"] = (
    error_df["predicted_probability"] >= 0.5
).astype(int)

error_df["error_type"] = np.select(
    [
        (error_df["predicted_label"] == 1) &
        (error_df["is_declining_label"] == 0),

        (error_df["predicted_label"] == 0) &
        (error_df["is_declining_label"] == 1),
    ],
    [
        "False positive",
        "False negative",
    ],
    default="Correct"
)

# Most confident mistakes first.
mistakes = (
    error_df[error_df["error_type"] != "Correct"]
    .assign(
        confidence=lambda d: np.where(
            d["error_type"].eq("False positive"),
            d["predicted_probability"],
            1 - d["predicted_probability"]
        )
    )
    .sort_values("confidence", ascending=False)
)

print("Number of test errors:", len(mistakes))
display(mistakes.head(10).round(4))


Number of test errors: 762


,content_id,client_id,is_declining_label,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,days_since_last_update,content_age_days,word_count,predicted_probability,predicted_label,error_type,confidence
5770,content_28b4223f4e5f,client_98a3ab7c34,1,1,0,1,0.00,0.0,1,91,3109.0,0.0799,0,False negative,0.9201
3879,content_34b14c00f80c,client_d4735e3a26,1,3,0,1,0.00,0.0,20,308,659.0,0.0822,0,False negative,0.9178
27177,content_79ac977c6e0b,client_f74efabef1,1,3,0,1,0.00,0.7,8,104,2304.0,0.1495,0,False negative,0.8505
22991,content_472ce7ae14c0,client_d4735e3a26,1,3,1,2,33.33,0.3,20,300,684.0,0.1522,0,False negative,0.8478
5608,content_a55d958ec725,client_d4735e3a26,1,3,0,1,0.00,2.7,20,290,837.0,0.1640,0,False negative,0.8360
12864,content_f1ef151d5e36,client_d4735e3a26,1,3,0,2,0.00,2.0,20,294,978.0,0.1659,0,False negative,0.8341
12076,content_230de4c50860,client_d4735e3a26,1,3,0,1,0.00,2.0,20,288,824.0,0.1698,0,False negative,0.8302
25838,content_cbc3b52a2ac1,client_98a3ab7c34,1,2,0,1,0.00,3.0,1,125,2286.0,0.1713,0,False negative,0.8287
13659,content_4c437dd8c1ee,client_d4735e3a26,1,3,0,1,0.00,3.0,20,284,840.0,0.1741,0,False negative,0.8259
23810,content_37804210415c,client_d4735e3a26,1,4,0,3,0.00,2.0,20,300,813.0,0.1748,0,False negative,0.8252


## Errors and interpretation

I inspect cases where the best model is highly confident but wrong.

A false positive means the model predicted a high probability of decline, but the observed label is not declining.

A false negative means the model assigned a low probability of decline even though the observed label is declining.

The model is decision-support rather than proof of why a page is declining. Strong predictions can still be affected by seasonality, content type, query mix, or limited historical data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.